In [ ]:
import os, sys, time, json, shutil, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from IPython.display import Audio, display, HTML
import librosa
import librosa.display
import soundfile as sf
from pathlib import Path
import scipy.signal as signal
from scipy.ndimage import uniform_filter1d
import pandas as pd
warnings.filterwarnings('ignore')

_espeak_dir = r"C:\Program Files\eSpeak NG"
if os.path.isdir(_espeak_dir) and _espeak_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = _espeak_dir + os.pathsep + os.environ.get("PATH", "")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

BASE_DIR = Path(".").resolve().parent
OUT_DIR  = BASE_DIR / "src" / "audio" / "lab3_results"
DATA_DIR = BASE_DIR / "audio" / "lab3_data"

for d in [OUT_DIR, DATA_DIR,
          OUT_DIR / "tacotron2", OUT_DIR / "vits",
          OUT_DIR / "finetuned_cfg1", OUT_DIR / "finetuned_cfg2",
          OUT_DIR / "training_run1",  OUT_DIR / "training_run2",
          OUT_DIR / "training_run3_finetune"]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
SENTENCES = [
    "The future of artificial intelligence represents a revolution in progress.",
    "Neural text-to-speech systems have fundamentally transformed human-computer interaction.",
    "Tacotron2 combines a sequence-to-sequence model with a neural vocoder architecture.",
    "The model learns to map text directly to mel-spectrograms during training.",
    "Deep learning has enabled unprecedented advances in speech synthesis quality.",
    "How does Tacotron2 achieve such remarkably natural-sounding speech?",
    "Are these systems already indistinguishable from human voice recordings?",
    "What comes next in the evolution of text-to-speech technology?",
    "The results are truly remarkable and exceed all prior expectations!",
    "VITS produces speech sometimes indistinguishable from human voice recordings!",
    "This breakthrough changed everything about voice interface design!",
    "The training pipeline has three stages: preprocessing, training, and evaluation.",
    "Two key components define the architecture: the posterior encoder and the flow-based prior.",
    "VITS — an end-to-end model — requires no separate vocoder during inference.",
    "One model, one forward pass — that is the promise of modern TTS systems.",
    "From rule-based systems to deep generative models: the field has come a long way.",
    "Applications span accessibility tools, virtual assistants, audiobooks, and education.",
    "The question is no longer whether neural TTS will succeed, but how far it will go.",
    "Speech synthesis has reached human parity on clean read speech benchmarks.",
    "We compare two architectures to understand their trade-offs in quality and speed.",
]


In [ ]:
tacotron2_files = [str(OUT_DIR / "tacotron2" / f"sent_{i:02d}.wav") for i in range(len(SENTENCES))]
vits_files      = [str(OUT_DIR / "vits"      / f"sent_{i:02d}.wav") for i in range(len(SENTENCES))]

from TTS.api import TTS as CoquiTTS
t2   = CoquiTTS("tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False, gpu=torch.cuda.is_available())
vits = CoquiTTS("tts_models/en/ljspeech/vits",          progress_bar=False, gpu=torch.cuda.is_available())

def compute_features(wav_path, sr=22050):
    y, _ = librosa.load(wav_path, sr=sr)
    duration  = len(y) / sr
    mfcc      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=14)[1:]
    centroid  = float(librosa.feature.spectral_centroid(y=y, sr=sr).mean())
    bandwidth = float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean())
    rolloff   = float(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85).mean())
    flatness  = float(librosa.feature.spectral_flatness(y=y).mean())
    contrast  = librosa.feature.spectral_contrast(y=y, sr=sr).mean(axis=1)
    f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"),
                                      sr=sr, frame_length=2048)
    voiced_f0    = f0[voiced_flag] if voiced_flag is not None else np.array([])
    f0_mean      = float(np.nanmean(voiced_f0)) if len(voiced_f0) > 0 else 0.0
    f0_std       = float(np.nanstd(voiced_f0))  if len(voiced_f0) > 0 else 0.0
    voiced_ratio = float(voiced_flag.mean())     if voiced_flag is not None else 0.0
    rms = float(librosa.feature.rms(y=y).mean())
    zcr = float(librosa.feature.zero_crossing_rate(y).mean())
    return {"duration": duration, "rms": rms, "zcr": zcr,
            "centroid": centroid, "bandwidth": bandwidth,
            "rolloff": rolloff, "flatness": flatness,
            "contrast_mean": float(contrast.mean()),
            "f0_mean": f0_mean, "f0_std": f0_std, "voiced_ratio": voiced_ratio,
            "mfcc_mean": mfcc.mean(axis=1).tolist(),
            "mfcc_std":  mfcc.std(axis=1).tolist(),
            "f0_series": f0.tolist() if f0 is not None else [],
            "voiced_flag": voiced_flag.tolist() if voiced_flag is not None else [],
            "audio_len": len(y), "sr": sr}

t2_feats   = [compute_features(f) for f in tacotron2_files]
vits_feats = [compute_features(f) for f in vits_files]
print("Признаки загружены")


## 7. Дообучение (Fine-tuning) Tacotron2

### 7.1 Подготовка датасета LJSpeech

LJSpeech-1.1 (~2.6 ГБ): 13 100 коротких записей чтения вслух (Linda Johnson, 22 050 Гц).

Ожидаемый путь: `audio/lab3_data/LJSpeech-1.1/`
Структура: `metadata.csv` + папка `wavs/`


In [ ]:
# Реальная структура датасета (48kHz + MossFormer2 super-resolution):
#   audio/lab3_data/LJSpeech-1.1/LJSpeech-1.1-48kHz/
#       metadata.csv
#       wavs/MossFormer2_SR_48K/*.wav   ← wav-файлы здесь, не напрямую в wavs/
LJSPEECH_ROOT = DATA_DIR / "LJSpeech-1.1" / "LJSpeech-1.1-48kHz"
LJSPEECH_WAVS = LJSPEECH_ROOT / "wavs" / "MossFormer2_SR_48K"
LJSPEECH_META = LJSPEECH_ROOT / "metadata.csv"

SUBSET_DIR  = DATA_DIR / "ljspeech_subset"
SUBSET_SIZE = 200


def ljspeech_48k_formatter(root_path, manifest_file, **kwargs):
    """Форматтер для LJSpeech-1.1-48kHz с MossFormer2 super-resolution.
    wav-файлы лежат в wavs/MossFormer2_SR_48K/, а не прямо в wavs/.
    AudioProcessor автоматически ресэмплирует 48kHz → 22050Hz при загрузке.
    """
    items = []
    meta_path = os.path.join(root_path, manifest_file)
    wav_dir   = os.path.join(root_path, "wavs", "MossFormer2_SR_48K")
    with open(meta_path, encoding="utf-8") as fh:
        for line in fh:
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            wav_id = parts[0].strip()
            text   = parts[2].strip() if len(parts) >= 3 else parts[1].strip()
            wav_path = os.path.join(wav_dir, wav_id + ".wav")
            if os.path.exists(wav_path):
                items.append({
                    "text":         text,
                    "audio_file":   wav_path,
                    "speaker_name": "ljspeech",
                    "root_path":    root_path,
                })
    return items


def check_ljspeech():
    if LJSPEECH_META.exists() and LJSPEECH_WAVS.exists():
        n = len(list(LJSPEECH_WAVS.glob("*.wav")))
        print(f"✓ LJSpeech найден: {n} wav-файлов (48kHz, MossFormer2)")
        print(f"  meta : {LJSPEECH_META}")
        print(f"  wavs : {LJSPEECH_WAVS}")
        return True
    print(f"✗ LJSpeech НЕ найден. Ожидается структура:")
    print(f"  {LJSPEECH_ROOT}/metadata.csv")
    print(f"  {LJSPEECH_WAVS}/*.wav")
    return False


HAS_LJS = check_ljspeech()
DO_TRAINING = HAS_LJS
print(f"\nDO_TRAINING = {DO_TRAINING}")


In [ ]:
if DO_TRAINING:
    subset_wav_dir = SUBSET_DIR / "wavs" / "MossFormer2_SR_48K"
    subset_wav_dir.mkdir(parents=True, exist_ok=True)

    with open(LJSPEECH_META, encoding="utf-8") as fh:
        all_lines = [ln.strip() for ln in fh if ln.strip()]

    subset_lines = all_lines[:SUBSET_SIZE]
    copied = 0
    for ln in subset_lines:
        wav_id = ln.split("|")[0]
        src = LJSPEECH_WAVS / f"{wav_id}.wav"
        dst = subset_wav_dir / f"{wav_id}.wav"
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    with open(SUBSET_DIR / "metadata.csv", "w", encoding="utf-8") as fh:
        fh.write("\n".join(subset_lines))

    print(f"Создан subset: {len(subset_lines)} строк, {copied} новых wav-файлов")
    print(f"Путь: {SUBSET_DIR}")
else:
    print("DO_TRAINING=False — пропуск создания subset")


### 7.2 Конфигурации обучения

| Параметр | Config 1 | Config 2 |
|----------|----------|----------|
| Learning rate | **1e-3** | **5e-4** |
| Batch size | **8** | **16** |
| Reduction factor r | **6** | **4** |
| Epochs | 30 | 30 |
| Mixed precision | да | да |


In [ ]:
if DO_TRAINING:
    from trainer import Trainer, TrainerArgs
    from TTS.tts.configs.tacotron2_config import Tacotron2Config
    from TTS.tts.configs.shared_configs import BaseDatasetConfig
    from TTS.tts.datasets import load_tts_samples
    from TTS.tts.models.tacotron2 import Tacotron2
    from TTS.tts.utils.text.tokenizer import TTSTokenizer
    from TTS.utils.audio import AudioProcessor

    # Регистрируем кастомный форматтер (48kHz, вложенный wavs/)
    import TTS.tts.datasets.formatters as _fmt_module
    _fmt_module.ljspeech_48k_formatter = ljspeech_48k_formatter

    DATASET_CFG = BaseDatasetConfig(
        formatter="ljspeech_48k_formatter",
        meta_file_train="metadata.csv",
        path=str(SUBSET_DIR),
    )

    _COMMON = dict(
        num_loader_workers=0,
        num_eval_loader_workers=0,
        run_eval=True,
        test_delay_epochs=-1,
        double_decoder_consistency=True,
        epochs=30,
        grad_clip=5.0,
        mixed_precision=True,
        use_phonemes=False,
        text_cleaner="phoneme_cleaners",
        print_step=10,
        save_step=200,
        save_n_checkpoints=2,
        save_checkpoints=True,
        cudnn_benchmark=False,
        datasets=[DATASET_CFG],
        test_sentences=[
            "Speech synthesis has come a long way.",
            "How natural does this sound to you?",
        ],
    )

    cfg1 = Tacotron2Config(
        batch_size=8,
        eval_batch_size=4,
        r=6,
        lr=1e-3,
        output_path=str(OUT_DIR / "training_run1"),
        **_COMMON,
    )
    cfg2 = Tacotron2Config(
        batch_size=16,
        eval_batch_size=8,
        r=4,
        lr=5e-4,
        output_path=str(OUT_DIR / "training_run2"),
        **_COMMON,
    )

    print("Config 1:", f"lr={cfg1.lr}, batch={cfg1.batch_size}, r={cfg1.r}")
    print("Config 2:", f"lr={cfg2.lr}, batch={cfg2.batch_size}, r={cfg2.r}")


### 7.3 Запуск обучения

> Ожидаемое время (RTX 4070 Ti, 200 сэмплов, 30 эпох): ~10–20 мин на конфиг.


In [ ]:
if DO_TRAINING:
    print("="*65)
    print("ЗАПУСК 1: lr=1e-3, batch=8, r=6")
    print("="*65)

    ap1 = AudioProcessor(**cfg1.audio.to_dict())
    tok1, cfg1 = TTSTokenizer.init_from_config(cfg1)
    tr1, ev1 = load_tts_samples(
        DATASET_CFG, eval_split=True,
        eval_split_max_size=cfg1.eval_split_max_size,
        eval_split_size=cfg1.eval_split_size,
        formatter=ljspeech_48k_formatter,
    )
    mdl1 = Tacotron2(cfg1, ap1, tok1, speaker_manager=None)
    run1 = Trainer(
        TrainerArgs(restore_path=None, skip_train_epoch=False),
        cfg1, output_path=str(OUT_DIR / "training_run1"),
        model=mdl1, train_samples=tr1, eval_samples=ev1,
    )
    t0 = time.time()
    run1.fit()
    print(f"\nЗапуск 1 завершён за {(time.time()-t0)/60:.1f} мин")
else:
    print("DO_TRAINING=False — запуск 1 пропущен")


In [ ]:
if DO_TRAINING:
    print("="*65)
    print("ЗАПУСК 2: lr=5e-4, batch=16, r=4")
    print("="*65)

    ap2 = AudioProcessor(**cfg2.audio.to_dict())
    tok2, cfg2 = TTSTokenizer.init_from_config(cfg2)
    tr2, ev2 = load_tts_samples(
        DATASET_CFG, eval_split=True,
        eval_split_max_size=cfg2.eval_split_max_size,
        eval_split_size=cfg2.eval_split_size,
        formatter=ljspeech_48k_formatter,
    )
    mdl2 = Tacotron2(cfg2, ap2, tok2, speaker_manager=None)
    run2 = Trainer(
        TrainerArgs(restore_path=None, skip_train_epoch=False),
        cfg2, output_path=str(OUT_DIR / "training_run2"),
        model=mdl2, train_samples=tr2, eval_samples=ev2,
    )
    t0 = time.time()
    run2.fit()
    print(f"\nЗапуск 2 завершён за {(time.time()-t0)/60:.1f} мин")
else:
    print("DO_TRAINING=False — запуск 2 пропущен")


### 7.4 Обучение с нуля vs Дообучение (сравнение)

Запуски 1 и 2 используют **случайную инициализацию** (`restore_path=None`).
Запуск 3 стартует от **предобученного чекпоинта** Coqui TTS Tacotron2-DDC.

| | Обучение с нуля | Дообучение (fine-tune) |
|--|:---------------:|:----------------------:|
| Инициализация весов | Случайные | Предобученные |
| Loss на первом шаге | ≈ 10–20 | ≈ 1–3 |
| Скорость конвергенции | Медленная | Быстрая |
| Кол-во нужных данных | Большой датасет | Небольшой датасет |
| Риск катастрофического забывания | Нет | Есть (низкий LR!) |


#### Запуск 3 — Дообучение от предобученного Tacotron2-DDC (lr=1e-4)


In [ ]:
if DO_TRAINING:
    from TTS.utils.manage import ModelManager

    # Получить путь к предобученному чекпоинту Coqui TTS
    _mgr = ModelManager()
    pretrained_ckpt, _, _ = _mgr.download_model("tts_models/en/ljspeech/tacotron2-DDC")
    print(f"Предобученный чекпоинт: {pretrained_ckpt}")

    (OUT_DIR / "training_run3_finetune").mkdir(parents=True, exist_ok=True)

    cfg3 = Tacotron2Config(
        batch_size=8,
        eval_batch_size=4,
        r=6,
        lr=1e-4,          # в 10× ниже, чем cfg1 — стандарт для fine-tuning
        output_path=str(OUT_DIR / "training_run3_finetune"),
        **_COMMON,
    )
    print("Config 3 (fine-tune от pretrained):",
          f"lr={cfg3.lr}, batch={cfg3.batch_size}, r={cfg3.r}")

    print("="*65)
    print("ЗАПУСК 3: Fine-tune от Coqui TTS Tacotron2-DDC, lr=1e-4")
    print("="*65)

    ap3 = AudioProcessor(**cfg3.audio.to_dict())
    tok3, cfg3 = TTSTokenizer.init_from_config(cfg3)
    tr3, ev3 = load_tts_samples(
        DATASET_CFG, eval_split=True,
        eval_split_max_size=cfg3.eval_split_max_size,
        eval_split_size=cfg3.eval_split_size,
        formatter=ljspeech_48k_formatter,
    )
    mdl3 = Tacotron2(cfg3, ap3, tok3, speaker_manager=None)
    run3 = Trainer(
        TrainerArgs(
            restore_path=pretrained_ckpt,  # ← ключевое отличие от Run 1/2
            skip_train_epoch=False,
        ),
        cfg3, output_path=str(OUT_DIR / "training_run3_finetune"),
        model=mdl3, train_samples=tr3, eval_samples=ev3,
    )
    t0 = time.time()
    run3.fit()
    print(f"\nЗапуск 3 завершён за {(time.time()-t0)/60:.1f} мин")
else:
    print("DO_TRAINING=False — запуск 3 (fine-tune) пропущен")


## 8. Визуализация метрик обучения

Логи читаются из TensorBoard-событий (.tfevents.*) из каталогов запусков.


In [ ]:
def parse_tb(run_dir):
    """Читает TensorBoard события из папки запуска."""
    try:
        from tensorboard.backend.event_processing import event_accumulator as ea_mod
        run_dir = Path(run_dir)
        events = sorted(run_dir.rglob("events.out.tfevents.*"))
        if not events:
            return {}
        acc = ea_mod.EventAccumulator(str(events[0]))
        acc.Reload()
        out = {}
        for tag in acc.Tags().get("scalars", []):
            evs = acc.Scalars(tag)
            out[tag] = {"steps": [e.step for e in evs], "values": [e.value for e in evs]}
        return out
    except Exception as exc:
        print(f"TensorBoard parse error: {exc}")
        return {}


def find_latest_subdir(base):
    subs = sorted(Path(base).glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True)
    return str(subs[0]) if subs else str(base)


if DO_TRAINING:
    tb1 = parse_tb(find_latest_subdir(OUT_DIR / "training_run1"))
    tb2 = parse_tb(find_latest_subdir(OUT_DIR / "training_run2"))
    # Run 3 may not exist if DO_TRAINING was False earlier
    _run3_base = OUT_DIR / "training_run3_finetune"
    tb3 = parse_tb(find_latest_subdir(_run3_base)) if _run3_base.exists() else {}
    print("Config 1 (с нуля)   метрики:", list(tb1.keys())[:6])
    print("Config 2 (с нуля)   метрики:", list(tb2.keys())[:6])
    print("Config 3 (finetune) метрики:", list(tb3.keys())[:6])

    METRIC_KEYS = [
        ("loss",                  "Total Loss"),
        ("loss_mel_postnet",      "Mel Postnet Loss"),
        ("loss_mel",              "Mel Loss"),
        ("loss_stop_tokens",      "Stop Token Loss"),
        ("loss_decoder_b",        "Decoder B Loss (DDC)"),
        ("loader_time",           "Loader Time"),
    ]

    n_plots = sum(1 for k, _ in METRIC_KEYS
                  if any(k in key for key in tb1) or any(k in key for key in tb2))
    n_plots = max(n_plots, 1)
    cols = 3
    rows = (n_plots + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
    fig.suptitle("Функции потерь: Config 1 (lr=1e-3) vs Config 2 (lr=5e-4)",
                 fontsize=13, fontweight="bold")
    ax_list = axes.flat if rows > 1 else [axes] if cols == 1 else axes.flat
    ax_iter = iter(ax_list)

    for search_key, title in METRIC_KEYS:
        k1 = next((k for k in tb1 if search_key in k), None)
        k2 = next((k for k in tb2 if search_key in k), None)
        if not k1 and not k2:
            continue
        ax = next(ax_iter)
        for k, tb, color, lbl in [(k1, tb1, "steelblue", "Config1 lr=1e-3"),
                                   (k2, tb2, "tomato",    "Config2 lr=5e-4")]:
            if k:
                steps  = np.array(tb[k]["steps"])
                values = np.array(tb[k]["values"])
                ax.plot(steps, values, color=color, lw=0.8, alpha=0.5)
                if len(values) > 5:
                    smooth = uniform_filter1d(values, size=max(1, len(values)//10))
                    ax.plot(steps, smooth, color=color, lw=2.0, label=lbl)
        ax.set_title(title)
        ax.set_xlabel("Шаг")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)

    # Hide unused axes
    for ax in ax_iter:
        ax.set_visible(False)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "training_loss_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Сохранено: {OUT_DIR / 'training_loss_curves.png'}")
else:
    print("DO_TRAINING=False — пропуск визуализации потерь")


In [ ]:
if DO_TRAINING:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("Динамика Learning Rate", fontsize=13)

    for ax, (tb, lbl, color) in zip(axes,
        [(tb1, "Config 1 (lr=1e-3)", "steelblue"),
         (tb2, "Config 2 (lr=5e-4)", "tomato")]):
        lr_key = next((k for k in tb if "lr" in k.lower()), None)
        if lr_key:
            ax.plot(tb[lr_key]["steps"], tb[lr_key]["values"], color=color, lw=1.5)
            ax.set_yscale("log")
        else:
            ax.text(0.5, 0.5, "LR не залогирован", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
        ax.set_title(lbl); ax.set_xlabel("Шаг"); ax.set_ylabel("LR")
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "lr_schedule.png"), dpi=150, bbox_inches="tight")
    plt.show()

else:
    # Теоретический NoamLR schedule
    def noam_lr(step, d=256, warmup=4000, base_lr=1.0):
        return base_lr * (d ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)

    steps = np.arange(1, 600)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("Теоретический LR schedule (Noam/WarmupDecay)", fontsize=13)

    for ax, (base, lbl, color) in zip(axes,
        [(1e-3, "Config 1 — lr=1e-3", "steelblue"),
         (5e-4, "Config 2 — lr=5e-4", "tomato")]):
        lrs = [noam_lr(s, base_lr=base * 256**0.5) for s in steps]
        ax.plot(steps, lrs, color=color, lw=2)
        ax.set_title(lbl); ax.set_xlabel("Шаг"); ax.set_ylabel("LR")
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "lr_schedule.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Сохранено: {OUT_DIR / 'lr_schedule.png'}  (теоретический)")


### 8.3 Сравнение: Обучение с нуля vs Дообучение


In [ ]:
if DO_TRAINING and tb3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("С нуля (Run1, Run2) vs Дообучение от предобученных весов (Run3)\n"
                 "Loss на первых N шагах", fontsize=12, fontweight="bold")

    COMPARE_TAGS = [
        ("loss",             "Total Loss"),
        ("loss_mel_postnet", "Mel Postnet Loss"),
        ("loss_mel",         "Mel Loss"),
    ]

    for ax, (tag_pat, title) in zip(axes, COMPARE_TAGS):
        for tb, lbl, color, ls in [
            (tb1, "С нуля — lr=1e-3",      "steelblue",   "-"),
            (tb2, "С нуля — lr=5e-4",      "tomato",      "--"),
            (tb3, "Fine-tune — lr=1e-4",   "forestgreen", "-."),
        ]:
            k = next((x for x in tb if tag_pat in x), None)
            if k:
                steps  = np.array(tb[k]["steps"])
                values = np.array(tb[k]["values"])
                ax.plot(steps, values, color=color, lw=0.5, alpha=0.3, linestyle=ls)
                if len(values) > 5:
                    sm = uniform_filter1d(values, size=max(1, len(values)//10))
                    ax.plot(steps, sm, color=color, lw=2.2, label=lbl, linestyle=ls)

        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Шаг")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "scratch_vs_finetune_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Сохранено: {OUT_DIR / 'scratch_vs_finetune_curves.png'}")

    # Таблица начального и конечного loss
    print("\nDynamic таблица Loss (первый / последний шаг):")
    print(f"  {'Запуск':<28} {'Loss[0]':>10} {'Loss[-1]':>10}")
    print("  " + "-"*50)
    for name, tb in [("С нуля cfg1 (lr=1e-3)", tb1),
                     ("С нуля cfg2 (lr=5e-4)", tb2),
                     ("Fine-tune cfg3 (lr=1e-4)", tb3)]:
        k = next((x for x in tb if "loss" in x and "mel" not in x and "stop" not in x), None)
        if k and tb[k]["values"]:
            v = tb[k]["values"]
            print(f"  {name:<28} {v[0]:>10.4f} {v[-1]:>10.4f}")

else:
    print("Run3 не запущен или DO_TRAINING=False")
    print()
    print("Ожидаемые результаты:")
    print("  С нуля (cfg1):      Loss[0] ≈ 12–18  →  Loss[-1] ≈ 4–7")
    print("  С нуля (cfg2):      Loss[0] ≈ 12–18  →  Loss[-1] ≈ 4–8")
    print("  Fine-tune (cfg3):   Loss[0] ≈ 1–3   →  Loss[-1] ≈ 0.8–2")
    print()
    print("Fine-tune стартует с уже низкого loss, т.к. веса уже обучены на LJSpeech!")


## 9. Эволюция Mel-спектрограммы в ходе обучения

Схематичная визуализация: как меняется спектрограмма от инициализации к конвергенции.


In [ ]:
np.random.seed(42)
N_FRAMES, N_MELS = 200, 80

def make_mel_stage(noise_level, signal_strength):
    """Имитирует mel-спектрограмму на определённом этапе обучения."""
    mel = np.random.randn(N_MELS, N_FRAMES) * noise_level
    silence = np.ones(N_FRAMES)
    silence[:15] = 0; silence[185:] = 0

    # Форманты (F1~5-20, F2~20-40, F3~40-55)
    for band, amp in [(slice(5, 20), 2.5), (slice(20, 40), 2.0), (slice(40, 55), 1.2)]:
        mel[band, :] += signal_strength * amp * silence[np.newaxis, :]

    # Вариации для реалистичности
    for t in range(0, N_FRAMES, 20):
        mel[:, max(0, t-2):t+2] *= 0.4 + 0.6 * signal_strength
    return mel


STAGES = [
    ("Шаг 0\n(случайная инициализация)", 1.2, 0.0),
    ("Шаг 100\n(начало обучения)",        0.7, 0.3),
    ("Шаг 500\n(формирование структуры)", 0.35, 0.65),
    ("Шаг 1000\n(улучшение чёткости)",    0.15, 0.88),
    ("Шаг 3000+\n(конвергенция)",         0.04, 1.0),
]

# ── Статичная фигура (5 стадий) ────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(STAGES), figsize=(22, 5))
fig.suptitle("Эволюция Mel-спектрограммы при обучении Tacotron2",
             fontsize=13, fontweight="bold")

for ax, (title, noise, sig) in zip(axes, STAGES):
    mel = make_mel_stage(noise, sig)
    im = ax.imshow(mel, aspect="auto", origin="lower", cmap="magma",
                   vmin=-3, vmax=4)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Кадры"); ax.set_ylabel("Mel-канал" if ax == axes[0] else "")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

plt.tight_layout()
fig.savefig(str(OUT_DIR / "mel_evolution.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {OUT_DIR / 'mel_evolution.png'}")

# ── Анимация ───────────────────────────────────────────────────────────────────
fig_anim, ax_anim = plt.subplots(figsize=(9, 4))
ax_anim.set_xlabel("Кадры"); ax_anim.set_ylabel("Mel-канал")

frames = []
for title, noise, sig in STAGES:
    mel = make_mel_stage(noise, sig)
    im  = ax_anim.imshow(mel, aspect="auto", origin="lower", cmap="magma",
                          vmin=-3, vmax=4, animated=True)
    ttl = ax_anim.text(0.5, 1.04, f"Эволюция Mel-спектрограммы: {title}",
                        transform=ax_anim.transAxes, ha="center",
                        fontsize=10, fontweight="bold", animated=True)
    frames.append([im, ttl])

ani = animation.ArtistAnimation(fig_anim, frames, interval=1200,
                                 blit=True, repeat_delay=800)
plt.tight_layout()
display(HTML(ani.to_jshtml()))
plt.close(fig_anim)
print("Анимация показана выше.")


## 10. Синтез после дообучения

Загружаем лучшие чекпоинты и генерируем аудио для тех же предложений.


In [ ]:
def find_best_ckpt(run_base):
    """Возвращает путь к best_model.pth или последнему .pth в каталоге запуска."""
    base = Path(run_base)
    best = sorted(base.rglob("best_model.pth"))
    if best:
        return str(best[-1])
    all_ckpts = sorted(base.rglob("*.pth"), key=lambda p: p.stat().st_mtime)
    return str(all_ckpts[-1]) if all_ckpts else None


if DO_TRAINING:
    from TTS.utils.synthesizer import Synthesizer

    run1_actual = find_latest_subdir(OUT_DIR / "training_run1")
    run2_actual = find_latest_subdir(OUT_DIR / "training_run2")

    ckpt1 = find_best_ckpt(run1_actual)
    ckpt2 = find_best_ckpt(run2_actual)
    print(f"Config 1 чекпоинт: {ckpt1}")
    print(f"Config 2 чекпоинт: {ckpt2}")

    ft_files1, ft_files2 = [], []
    for ckpt, ft_dir, ft_list, cfg_label in [
        (ckpt1, OUT_DIR / "finetuned_cfg1", ft_files1, "Config1"),
        (ckpt2, OUT_DIR / "finetuned_cfg2", ft_files2, "Config2"),
    ]:
        if not ckpt:
            print(f"Чекпоинт для {cfg_label} не найден"); continue
        cfg_path = str(Path(ckpt).parent / "config.json")
        syn = Synthesizer(
            tts_checkpoint=ckpt,
            tts_config_path=cfg_path,
            use_cuda=torch.cuda.is_available(),
        )
        for i, sent in enumerate(SENTENCES):
            out = ft_dir / f"sent_{i:02d}.wav"
            wavs = syn.tts(sent)
            syn.save_wav(wavs, str(out))
            ft_list.append(str(out))
        print(f"{cfg_label}: сгенерировано {len(ft_list)} файлов")
else:
    # Для демонстрации используем предобученные файлы как «после»
    ft_files1 = tacotron2_files
    ft_files2 = vits_files
    print("DO_TRAINING=False — используем предобученные T2 и VITS для иллюстрации")


## 11. Сравнение «до» и «после» дообучения


In [ ]:
if DO_TRAINING:
    model_groups = [
        ("Tacotron2 (до дообучения)",     tacotron2_files, "Purples"),
        ("Tacotron2 (Config1, дообучен)", ft_files1,        "Blues"),
        ("Tacotron2 (Config2, дообучен)", ft_files2,        "Greens"),
    ]
else:
    model_groups = [
        ("Tacotron2 (предобуч.)", tacotron2_files, "Purples"),
        ("VITS (предобуч.)",      vits_files,      "Oranges"),
    ]

EX = [0, 5, 8]  # neutral, question, exclamation
EX_LBL = ["Нейтральное", "Вопрос", "Восклицание"]

fig, axes = plt.subplots(len(model_groups), len(EX), figsize=(len(EX)*7, len(model_groups)*4))
fig.suptitle("Мел-спектрограммы: сравнение вариантов модели",
             fontsize=13, fontweight="bold")

for row, (model_name, files, cmap) in enumerate(model_groups):
    for col, (idx, lbl) in enumerate(zip(EX, EX_LBL)):
        ax = axes[row, col] if len(model_groups) > 1 else axes[col]
        y, sr = librosa.load(files[idx], sr=22050)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, hop_length=256)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, sr=sr, hop_length=256,
                                        x_axis="time", y_axis="mel", ax=ax, cmap=cmap)
        ax.set_title(f"{model_name}\n{lbl}", fontsize=8)
        plt.colorbar(img, ax=ax, fraction=0.04)

plt.tight_layout()
fig.savefig(str(OUT_DIR / "before_after_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {OUT_DIR / 'before_after_comparison.png'}")


In [ ]:
# Акустические метрики «до» и «после»
print("\nАкустические признаки До vs После дообучения:")
print("-"*65)

all_sets = {}
if DO_TRAINING and ft_files1:
    all_sets["Tacotron2 (до)"]     = tacotron2_files
    all_sets["Tacotron2 (Cfg1)"]   = ft_files1
    all_sets["Tacotron2 (Cfg2)"]   = ft_files2
else:
    all_sets["Tacotron2 (pretrain)"] = tacotron2_files
    all_sets["VITS (pretrain)"]      = vits_files

comp_rows = []
for lbl, files in all_sets.items():
    feats = [compute_features(f) for f in files]
    comp_rows.append({
        "Модель":          lbl,
        "Avg Duration (s)": np.mean([f["duration"] for f in feats]),
        "Avg F0 (Hz)":      np.mean([f["f0_mean"]  for f in feats if f["f0_mean"] > 0]),
        "σ F0 (Hz)":        np.mean([f["f0_std"]   for f in feats]),
        "Voiced Ratio":     np.mean([f["voiced_ratio"] for f in feats]),
        "Spec. Centroid":   np.mean([f["centroid"]  for f in feats]),
    })

df_comp = pd.DataFrame(comp_rows).set_index("Модель").round(3)
display(df_comp)


## 12. Итоговое сравнение моделей


In [ ]:
# Скорость синтеза
print("Скорость синтеза (benchmark на предложении #0):")
for name, tts_obj, files in [("Tacotron2", t2,   tacotron2_files),
                               ("VITS",      vits, vits_files)]:
    times = []
    for _ in range(3):
        t0 = time.time()
        tts_obj.tts_to_file(text=SENTENCES[0],
                             file_path=str(OUT_DIR / f"_bench_{name}.wav"))
        times.append(time.time() - t0)
    audio_dur = compute_features(files[0])["duration"]
    rtf = np.mean(times) / audio_dur
    print(f"  {name:<12} | {np.mean(times):.3f}с синтез | audio={audio_dur:.2f}с | RTF={rtf:.3f}")

print()
(OUT_DIR / "_bench_Tacotron2.wav").unlink(missing_ok=True)
(OUT_DIR / "_bench_VITS.wav").unlink(missing_ok=True)

# Финальная сводная таблица
summary = pd.DataFrame({
    "Параметр": [
        "Тип архитектуры",
        "Отдельный вокодер",
        "Год публикации",
        "Trainable параметры (approx.)",
        "Avg Duration (s)",
        "Avg F0 mean (Hz)",
        "Avg F0 std (Hz)",
        "Voiced Ratio",
        "Spec. Centroid (Hz)",
        "MOS (LJSpeech, оценочно)",
        "Attention failures",
    ],
    "Tacotron2-DDC": [
        "Seq2Seq + Vocoder",
        "Да (WaveGlow / G-L)",
        "2018",
        "~28M",
        f"{np.mean([f['duration'] for f in t2_feats]):.2f}",
        f"{np.mean([f['f0_mean'] for f in t2_feats if f['f0_mean']>0]):.1f}",
        f"{np.mean([f['f0_std'] for f in t2_feats]):.1f}",
        f"{np.mean([f['voiced_ratio'] for f in t2_feats]):.3f}",
        f"{np.mean([f['centroid'] for f in t2_feats]):.0f}",
        "~4.0",
        "Возможны",
    ],
    "VITS": [
        "End-to-end VAE+GAN",
        "Нет",
        "2021",
        "~83M",
        f"{np.mean([f['duration'] for f in vits_feats]):.2f}",
        f"{np.mean([f['f0_mean'] for f in vits_feats if f['f0_mean']>0]):.1f}",
        f"{np.mean([f['f0_std'] for f in vits_feats]):.1f}",
        f"{np.mean([f['voiced_ratio'] for f in vits_feats]):.3f}",
        f"{np.mean([f['centroid'] for f in vits_feats]):.0f}",
        "~4.4",
        "Нет (без attention)",
    ],
})
display(summary.set_index("Параметр"))


In [ ]:
# Сравнение спектральных контрастов по типам интонации
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Mel-спектрограммы по типам интонации\nTacotron2 (верхний ряд) vs VITS (нижний ряд)",
             fontsize=13, fontweight="bold")

INTON = [(0, "Нейтральное"), (5, "Вопрос"), (8, "Восклицание")]

for col, (idx, lbl) in enumerate(INTON):
    for row, (files, model, cmap) in enumerate([
        (tacotron2_files, "Tacotron2", "magma"),
        (vits_files,      "VITS",      "inferno"),
    ]):
        ax = axes[row, col]
        y, sr = librosa.load(files[idx], sr=22050)
        mel   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, hop_length=256)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, sr=sr, hop_length=256,
                                        x_axis="time", y_axis="mel",
                                        ax=ax, cmap=cmap)
        ax.set_title(f"{model}\n{lbl}: «{SENTENCES[idx][:38]}…»", fontsize=8)
        plt.colorbar(img, ax=ax, fraction=0.04)

plt.tight_layout()
fig.savefig(str(OUT_DIR / "intonation_mel_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {OUT_DIR / 'intonation_mel_comparison.png'}")


## 13. VITS на русском языке: обучение с нуля на RUSLAN

### Почему нельзя просто дообучить английскую модель?

Английские Tacotron2/VITS обучены на **латинских фонемах**.
Русский — кириллица + другой фонемный инвентарь (palatalization, etc.).
Текстовый энкодер просто **не знает** эти символы → нужно обучать с нуля.

### RUSLAN датасет
- Мужской диктор (Руслан Савченко), ~31 ч, 22 050 Гц
- Сайт: https://ruslan-corpus.github.io/
- Формат: `metadata.csv` (`id|text|normalized`) + папка `wavs/`
- Ожидаемый путь: `audio/lab3_data/RUSLAN/RUSLAN/`

### Архитектура VITS (напоминание)
VITS не требует отдельного вокодера — один forward pass:
```
Текст (ru phonemes) → Text Encoder → prior z ~ N(0,I)
                                          ↓ Flows
                                    HiFi-GAN → audio
```
Дискриминаторы MPD + MSD дообучают вокодерную часть adversarially.


In [ ]:
RUSLAN_DIR  = DATA_DIR / "RUSLAN" / "RUSLAN"
RUSLAN_META = RUSLAN_DIR / "metadata.csv"

def check_ruslan():
    if RUSLAN_META.exists():
        n = len(list((RUSLAN_DIR / "wavs").glob("*.wav")))
        print(f"✓ RUSLAN: {n} wav-файлов  →  {RUSLAN_DIR}")
        return True
    print("✗ RUSLAN не найден.")
    print("  Скачать:")
    print("    https://ruslan-corpus.github.io/  (требует регистрации)")
    print(f"  Распаковать в: {RUSLAN_DIR}/")
    print("  Ожидаемая структура:")
    print("    RUSLAN/metadata.csv")
    print("    RUSLAN/wavs/*.wav")
    return False

HAS_RUSLAN = check_ruslan()
DO_VITS_RU = HAS_RUSLAN  # поставить True вручную после скачивания
print(f"\nDO_VITS_RU = {DO_VITS_RU}")


In [ ]:
def ruslan_formatter(root_path, manifest_file, **kwargs):
    """Читает metadata.csv RUSLAN и возвращает список сэмплов для Coqui TTS."""
    items = []
    wav_dir = os.path.join(root_path, "wavs")
    meta_path = os.path.join(root_path, manifest_file)
    with open(meta_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split("|")
            if len(parts) < 2:
                continue
            wav_id = parts[0].strip()
            # Используем нормализованный текст (3-я колонка) если есть
            text = parts[2].strip() if len(parts) >= 3 else parts[1].strip()
            wav_path = os.path.join(wav_dir, wav_id + ".wav")
            if not os.path.exists(wav_path):
                wav_path = os.path.join(wav_dir, wav_id)  # без расширения
            if os.path.exists(wav_path):
                items.append({
                    "text":         text,
                    "audio_file":   wav_path,
                    "speaker_name": "ruslan",
                    "root_path":    root_path,
                    "language":     "ru",
                })
    print(f"ruslan_formatter: загружено {len(items)} сэмплов из {meta_path}")
    return items


if DO_VITS_RU:
    # Быстрая проверка форматтера
    sample = ruslan_formatter(str(RUSLAN_DIR), "metadata.csv")
    if sample:
        print(f"Пример: {sample[0]}")
else:
    print("DO_VITS_RU=False — форматтер определён, но не применяется")


### 13.1 Конфигурация VITS для русского языка

Ключевые отличия от английской конфигурации:

| Параметр | Значение | Причина |
|----------|----------|---------|
| `phoneme_language` | `"ru"` | espeak-ng транслитерирует кириллицу в IPA |
| `text_cleaner` | `"phoneme_cleaners"` | очистка + G2P для русского |
| `use_phonemes` | `True` | фонемы вместо сырых символов |
| `lr_gen / lr_disc` | `2e-4` | стандарт VITS |
| `batch_size` | `16` | для RTX 4070 Ti (12 ГБ) |

> espeak-ng уже установлен (`winget install eSpeak-NG.eSpeak-NG`) и поддерживает русский.


In [ ]:
if DO_VITS_RU:
    from trainer import Trainer, TrainerArgs
    from TTS.tts.configs.vits_config import VitsConfig
    from TTS.tts.models.vits import Vits, VitsAudioConfig
    from TTS.tts.datasets import load_tts_samples
    from TTS.tts.utils.text.tokenizer import TTSTokenizer
    from TTS.utils.audio import AudioProcessor
    from TTS.tts.configs.shared_configs import BaseDatasetConfig

    RUSLAN_DATASET_CFG = BaseDatasetConfig(
        formatter="ruslan_formatter",   # наш кастомный форматтер
        meta_file_train="metadata.csv",
        path=str(RUSLAN_DIR),
        language="ru",
    )

    vits_ru_audio = VitsAudioConfig(
        sample_rate=22050,
        win_length=1024,
        hop_length=256,
        num_mels=80,
        mel_fmin=0,
        mel_fmax=None,
    )

    vits_ru_cfg = VitsConfig(
        audio=vits_ru_audio,
        run_name="vits_ruslan_scratch",
        batch_size=16,
        eval_batch_size=8,
        batch_group_size=5,
        num_loader_workers=4,
        num_eval_loader_workers=4,
        run_eval=True,
        test_delay_epochs=5,
        epochs=1000,
        # ── Русский язык ──────────────────────────────
        text_cleaner="phoneme_cleaners",
        use_phonemes=True,
        phoneme_language="ru",
        phoneme_cache_path=str(OUT_DIR / "phoneme_cache_ru"),
        compute_input_seq_cache=True,
        # ── Оптимизация ───────────────────────────────
        lr_gen=2e-4,
        lr_disc=2e-4,
        mixed_precision=True,
        print_step=50,
        save_step=1000,
        save_n_checkpoints=3,
        save_checkpoints=True,
        # ── Пути ──────────────────────────────────────
        output_path=str(OUT_DIR / "vits_ruslan_scratch"),
        datasets=[RUSLAN_DATASET_CFG],
        cudnn_benchmark=False,
        test_sentences=[
            "Привет! Это система синтеза речи на основе нейросетевых моделей.",
            "Как вы себя чувствуете сегодня?",
            "Нейронные сети изменили всё: от распознавания речи до её синтеза!",
        ],
    )

    print("VITS RU конфиг:")
    print(f"  phoneme_language : {vits_ru_cfg.phoneme_language}")
    print(f"  use_phonemes     : {vits_ru_cfg.use_phonemes}")
    print(f"  batch_size       : {vits_ru_cfg.batch_size}")
    print(f"  lr_gen / lr_disc : {vits_ru_cfg.lr_gen} / {vits_ru_cfg.lr_disc}")
    print(f"  epochs           : {vits_ru_cfg.epochs}")
    print(f"  output           : {vits_ru_cfg.output_path}")
else:
    print("DO_VITS_RU=False — конфиг пропущен")


### 13.2 Запуск обучения

> **Ожидаемое время до разборчивой речи:**
> - RTX 4070 Ti: ~4–6 ч до первых читаемых фраз (~5 000 шагов)
> - Хорошее качество: ~50 000+ шагов (~24–48 ч)
> - Пока идёт обучение — смотри TensorBoard (см. ячейку ниже)


In [ ]:
if DO_VITS_RU:
    (OUT_DIR / "vits_ruslan_scratch").mkdir(parents=True, exist_ok=True)

    # Регистрируем кастомный форматтер в Coqui TTS
    from TTS.tts.datasets import formatters as _fmt_module
    _fmt_module.ruslan_formatter = ruslan_formatter

    ap_ru = AudioProcessor(**vits_ru_cfg.audio.to_dict())
    tok_ru, vits_ru_cfg = TTSTokenizer.init_from_config(vits_ru_cfg)

    train_ru, eval_ru = load_tts_samples(
        RUSLAN_DATASET_CFG,
        eval_split=True,
        eval_split_max_size=500,
        eval_split_size=0.01,
        formatter=ruslan_formatter,      # передаём явно
    )
    print(f"Train: {len(train_ru)} | Eval: {len(eval_ru)}")

    mdl_ru = Vits(vits_ru_cfg, ap_ru, tok_ru, speaker_manager=None)
    print(f"Параметров модели: {sum(p.numel() for p in mdl_ru.parameters()):,}")

    trainer_ru = Trainer(
        TrainerArgs(
            restore_path=None,           # ← с нуля, без предобученных весов
            skip_train_epoch=False,
        ),
        vits_ru_cfg,
        output_path=str(OUT_DIR / "vits_ruslan_scratch"),
        model=mdl_ru,
        train_samples=train_ru,
        eval_samples=eval_ru,
    )

    print("\nЗапуск обучения VITS на RUSLAN (с нуля)…")
    print("Для мониторинга в отдельном терминале:")
    print(f"  tensorboard --logdir {OUT_DIR / 'vits_ruslan_scratch'}")

    t0 = time.time()
    trainer_ru.fit()
    elapsed = time.time() - t0
    print(f"\nОбучение завершено за {elapsed/3600:.1f} ч ({elapsed/60:.0f} мин)")
else:
    print("DO_VITS_RU=False — обучение пропущено")
    print()
    print("Примерный ход обучения VITS на русском (RTX 4070 Ti):")
    print("  Шаг     0: loss_gen ≈ 30–50, loss_disc ≈ 3–8  (хаос)")
    print("  Шаг  2000: loss_gen ≈ 10–20, начинают угадываться фонемы")
    print("  Шаг  5000: первые разборчивые слова")
    print("  Шаг 20000: стабильная разборчивость, артефакты ещё есть")
    print("  Шаг 50000: качество близко к LJSpeech-уровню")


In [ ]:
# ── Мониторинг обучения через TensorBoard ─────────────────────────────────────
# Можно запустить прямо из ноутбука (параллельно с обучением)

vits_ru_tb_dir = OUT_DIR / "vits_ruslan_scratch"

if vits_ru_tb_dir.exists() and list(vits_ru_tb_dir.rglob("events.out.tfevents.*")):
    tb_ru = parse_tb(find_latest_subdir(vits_ru_tb_dir))
    print("Доступные метрики VITS RU:", list(tb_ru.keys())[:12])

    VITS_METRICS = [
        ("loss_0",          "Generator Loss (total)"),
        ("loss_1",          "Discriminator Loss"),
        ("loss_mel",        "Mel Loss"),
        ("loss_kl",         "KL Divergence Loss"),
        ("loss_feat",       "Feature Matching Loss"),
        ("loss_duration",   "Duration Loss"),
    ]

    n_found = sum(1 for k, _ in VITS_METRICS
                  if any(k in key for key in tb_ru))
    cols = min(3, n_found) if n_found > 0 else 3
    rows = max(1, (n_found + cols - 1) // cols)

    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
    fig.suptitle("VITS (RUSLAN, с нуля): кривые потерь", fontsize=13, fontweight="bold")
    ax_it = iter(np.array(axes).flatten())

    for search_k, title in VITS_METRICS:
        key = next((k for k in tb_ru if search_k in k), None)
        if not key:
            continue
        ax = next(ax_it)
        steps  = np.array(tb_ru[key]["steps"])
        values = np.array(tb_ru[key]["values"])
        ax.plot(steps, values, lw=0.6, alpha=0.35, color="mediumpurple")
        if len(values) > 10:
            sm = uniform_filter1d(values, size=max(1, len(values)//15))
            ax.plot(steps, sm, lw=2.2, color="mediumpurple", label=title)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("Шаг")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)

    for ax in ax_it:
        ax.set_visible(False)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "vits_ru_loss_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Сохранено: {OUT_DIR / 'vits_ru_loss_curves.png'}")

else:
    # Теоретические кривые VITS (GAN)
    np.random.seed(7)
    steps = np.arange(0, 50001, 100)

    def smooth_curve(start, end, noise=0.05, n=len(steps)):
        x = np.linspace(0, 1, n)
        base = start * np.exp(-3 * x) + end
        return base + np.random.randn(n) * noise * start

    curves = {
        "Generator Loss":     smooth_curve(40,  3.5, 0.12),
        "Discriminator Loss": smooth_curve(6,   1.2, 0.08),
        "Mel Loss":           smooth_curve(2.5, 0.4, 0.05),
        "KL Divergence":      smooth_curve(5,   0.8, 0.06),
        "Feature Matching":   smooth_curve(3,   0.5, 0.04),
        "Duration Loss":      smooth_curve(1.5, 0.2, 0.03),
    }

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle("VITS (RUSLAN, с нуля): ожидаемый вид кривых потерь\n"
                 "(схематично, реальные значения после запуска обучения)",
                 fontsize=12, fontweight="bold")

    for ax, (title, vals) in zip(axes.flat, curves.items()):
        ax.plot(steps, vals, lw=0.5, alpha=0.3, color="mediumpurple")
        sm = uniform_filter1d(vals, size=15)
        ax.plot(steps, sm, lw=2.2, color="mediumpurple")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Шаг")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "vits_ru_loss_curves_theoretical.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Показаны теоретические кривые (до запуска обучения)")
    print(f"Сохранено: {OUT_DIR / 'vits_ru_loss_curves_theoretical.png'}")


### 13.3 Синтез русской речи после обучения


In [ ]:
RU_TEST_SENTENCES = [
    # Нейтральные
    "Нейронные сети изменили подход к синтезу речи.",
    "Модель обучена на наборе данных RUSLAN с нуля.",
    # Вопросы
    "Как звучит синтезированная речь на русском языке?",
    "Что такое вариационный автоэнкодер в контексте VITS?",
    # Восклицания
    "Результаты превзошли все ожидания!",
    "Это действительно работает — синтез с нуля!",
    # Двоеточие и тире
    "Архитектура включает три компонента: энкодер, потоки и декодер.",
    "VITS — сквозная модель — не требует отдельного вокодера.",
]

vits_ru_out_dir = OUT_DIR / "vits_ruslan_synth"
vits_ru_out_dir.mkdir(parents=True, exist_ok=True)

ru_ckpt = find_best_ckpt(OUT_DIR / "vits_ruslan_scratch") if DO_VITS_RU else None

if ru_ckpt:
    from TTS.utils.synthesizer import Synthesizer

    run_dir = find_latest_subdir(OUT_DIR / "vits_ruslan_scratch")
    syn_ru = Synthesizer(
        tts_checkpoint=ru_ckpt,
        tts_config_path=str(Path(run_dir) / "config.json"),
        use_cuda=torch.cuda.is_available(),
    )

    ru_files = []
    for i, sent in enumerate(RU_TEST_SENTENCES):
        out_p = str(vits_ru_out_dir / f"ru_{i:02d}.wav")
        wavs = syn_ru.tts(sent)
        syn_ru.save_wav(wavs, out_p)
        ru_files.append(out_p)
        print(f"  [{i+1}] {sent[:55]}…")

    print(f"\nСгенерировано {len(ru_files)} файлов")

    # Прослушивание
    for i, (f, sent) in enumerate(zip(ru_files, RU_TEST_SENTENCES)):
        print(f"\n{sent}")
        display(Audio(f, rate=22050))

    # Mel-спектрограммы русской речи
    fig, axes = plt.subplots(2, 4, figsize=(22, 8))
    fig.suptitle("Mel-спектрограммы: VITS (RUSLAN, с нуля)", fontsize=13, fontweight="bold")
    for ax, (f, sent) in zip(axes.flat, zip(ru_files, RU_TEST_SENTENCES)):
        y, sr = librosa.load(f, sr=22050)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80, hop_length=256)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(mel_db, sr=sr, hop_length=256,
                                  x_axis="time", y_axis="mel", ax=ax, cmap="magma")
        ax.set_title(sent[:35] + "…", fontsize=7)
    plt.tight_layout()
    fig.savefig(str(OUT_DIR / "vits_ru_mel_spectrograms.png"), dpi=150, bbox_inches="tight")
    plt.show()

else:
    print("Чекпоинт VITS RU не найден (обучение ещё не запускалось или DO_VITS_RU=False).")
    print()
    print("Тестовые предложения для синтеза после обучения:")
    for i, s in enumerate(RU_TEST_SENTENCES):
        print(f"  [{i+1}] {s}")


### Быстрый инференс: VITS (RUSLAN)
Ячейка работает независимо от флага `DO_VITS_RU` — нужен только готовый чекпоинт.
Укажи свой текст в `CUSTOM_SENTENCES` и запусти.


In [ ]:
import glob, os
from pathlib import Path
from IPython.display import display, Audio
import soundfile as sf

# ── Настройки ─────────────────────────────────────────────────────────────────
RUSLAN_CKPT_DIR = OUT_DIR / "vits_ruslan_scratch"   # папка с чекпоинтами

# Свои предложения для синтеза (редактируй здесь)
CUSTOM_SENTENCES = [
    "Привет! Это синтез речи на русском языке.",
    "Нейронные сети изменили подход к синтезу.",
    "Как ты себя чувствуешь сегодня?",
    "Архитектура VITS не требует отдельного вокодера.",
]

# ── Поиск чекпоинта ────────────────────────────────────────────────────────────
def find_ckpt(base: Path):
    """Возвращает путь к best_model.pth или последнему checkpoint_*.pth."""
    best = list(base.rglob("best_model.pth"))
    if best:
        return str(sorted(best, key=lambda p: p.stat().st_mtime)[-1])
    ckpts = list(base.rglob("checkpoint_*.pth"))
    if ckpts:
        return str(sorted(ckpts, key=lambda p: p.stat().st_mtime)[-1])
    return None

ckpt_path = find_ckpt(RUSLAN_CKPT_DIR)

if ckpt_path is None:
    print("Чекпоинт не найден в:", RUSLAN_CKPT_DIR)
    print("Запусти обучение (start_training.ps1) и подожди хотя бы 1000 шагов.")
else:
    print(f"Чекпоинт: {ckpt_path}")

    # config.json лежит рядом с чекпоинтом
    cfg_path = str(Path(ckpt_path).parent / "config.json")
    if not os.path.exists(cfg_path):
        # иногда config.json на уровень выше
        cfg_path = str(Path(ckpt_path).parent.parent / "config.json")

    import torch
    from TTS.utils.synthesizer import Synthesizer

    syn = Synthesizer(
        tts_checkpoint=ckpt_path,
        tts_config_path=cfg_path,
        use_cuda=torch.cuda.is_available(),
    )
    out_sr = syn.output_sample_rate
    print(f"Модель загружена  (CUDA: {torch.cuda.is_available()},  output_sample_rate={out_sr})")

    out_dir = OUT_DIR / "vits_ruslan_listen"
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, text in enumerate(CUSTOM_SENTENCES):
        out_path = str(out_dir / f"listen_{i:02d}.wav")
        wavs = syn.tts(text)
        syn.save_wav(wavs, out_path)

        # Играем напрямую из numpy — минуем возможный rate-мискматч при чтении файла
        y = np.array(wavs, dtype=np.float32)
        y_norm = y / (np.max(np.abs(y)) + 1e-8)

        print(f"\n{text}")
        print(f"  samples={len(y)}, rate={out_sr}, dur={len(y)/out_sr:.2f}s, max_amp={np.max(np.abs(y)):.4f}")
        display(Audio(y_norm, rate=out_sr))


In [ ]:
# Сравнение акустических признаков: VITS английский vs VITS русский
if ru_ckpt and vits_files:
    print("Сравнение акустических признаков: VITS EN (LJSpeech) vs VITS RU (RUSLAN)\n")

    en_feats_vits = [compute_features(f) for f in vits_files[:8]]
    ru_feats_vits = [compute_features(f) for f in
                     sorted(vits_ru_out_dir.glob("*.wav"))[:8]]

    comp = []
    for lbl, feats in [("VITS EN (LJSpeech)", en_feats_vits),
                        ("VITS RU (RUSLAN)",   ru_feats_vits)]:
        comp.append({
            "Модель":      lbl,
            "Avg Dur (s)": np.mean([f["duration"]    for f in feats]),
            "F0 mean (Hz)":np.mean([f["f0_mean"]     for f in feats if f["f0_mean"] > 0]),
            "F0 std (Hz)": np.mean([f["f0_std"]      for f in feats]),
            "Voiced":      np.mean([f["voiced_ratio"] for f in feats]),
            "Centroid":    np.mean([f["centroid"]    for f in feats]),
        })

    display(pd.DataFrame(comp).set_index("Модель").round(3))
else:
    print("Для сравнения EN vs RU запустите обучение VITS RU")
